<a href="https://colab.research.google.com/github/OldManFly/OldManFly.github.io/blob/master/swinUnetr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. 安裝依賴套件
!pip install -q monai nibabel scikit-learn tqdm

# 2. 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 檢查 GPU 狀態
!nvidia-smi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 49.4 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tue Mar 24 08:48:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   27C    P0         

In [4]:
import os
import glob
import nibabel as nib
import numpy as np
from tqdm import tqdm

def analyze_mri_shapes(data_dir):
    """
    掃描目錄中的 MRI 影像並統計其形狀 (X, Y, Z) 的分佈。
    """
    # 尋找所有的 .nii 和 .nii.gz 檔案
    image_paths_gz = glob.glob(os.path.join(data_dir, "**", "*.nii.gz"), recursive=True)
    image_paths_nii = glob.glob(os.path.join(data_dir, "**", "*.nii"), recursive=True)
    all_image_paths = sorted(list(set(image_paths_gz + image_paths_nii)))

    if not all_image_paths:
        print(f"在 {data_dir} 找不到任何 .nii 或 .nii.gz 檔案。")
        return

    print(f"找到 {len(all_image_paths)} 個影像檔案，正在分析其尺寸...")

    shapes = []
    for img_path in tqdm(all_image_paths, desc="讀取影像中"):
        try:
            img = nib.load(img_path)
            # 取得影像的空間維度 (通常是前三個維度)
            shape = img.shape[:3]
            shapes.append(shape)
        except Exception as e:
            print(f"無法讀取 {img_path}: {e}")

    if not shapes:
        print("無法成功讀取任何影像形狀。")
        return

    shapes = np.array(shapes)

    print("\n--- 影像尺寸統計結果 ---")
    print(f"最小尺寸 (X, Y, Z): {np.min(shapes, axis=0)}")
    print(f"最大尺寸 (X, Y, Z): {np.max(shapes, axis=0)}")
    print(f"平均尺寸 (X, Y, Z): {np.mean(shapes, axis=0).astype(int)}")
    print(f"中位數尺寸 (X, Y, Z): {np.median(shapes, axis=0).astype(int)}")

    # 計算常見的百分位數，幫助決定 img_size
    print(f"第 25 百分位數: {np.percentile(shapes, 25, axis=0).astype(int)}")
    print(f"第 75 百分位數: {np.percentile(shapes, 75, axis=0).astype(int)}")
    print(f"第 90 百分位數: {np.percentile(shapes, 90, axis=0).astype(int)}")
    print("-------------------------")
    print("\n建議：")
    print("1. Swin UNETR 的 img_size 必須是 32 的倍數 (例如: 96, 128, 160)。")
    print("2. 如果記憶體允許，可以參考 '中位數' 或 '第 75 百分位數' 附近的尺寸，並向上/向下調整為 32 的倍數。")
    print("3. 如果影像尺寸差異很大，你可能需要在 training transforms 中設定適當的 RandCropByPosNegLabeld 尺寸。")

# 設定資料路徑並執行分析
data_directory = "/content/drive/MyDrive/swin_spine/trainData"
analyze_mri_shapes(data_directory)


找到 854 個影像檔案，正在分析其尺寸...


讀取影像中: 100%|██████████| 854/854 [04:24<00:00,  3.23it/s]


--- 影像尺寸統計結果 ---
最小尺寸 (X, Y, Z): [  8 305  12]
最大尺寸 (X, Y, Z): [1024 1168 3682]
平均尺寸 (X, Y, Z): [438 651 271]
中位數尺寸 (X, Y, Z): [512 640 120]
第 25 百分位數: [ 24 448  12]
第 75 百分位數: [880 880 448]
第 90 百分位數: [880 880 611]
-------------------------

建議：
1. Swin UNETR 的 img_size 必須是 32 的倍數 (例如: 96, 128, 160)。
2. 如果記憶體允許，可以參考 '中位數' 或 '第 75 百分位數' 附近的尺寸，並向上/向下調整為 32 的倍數。
3. 如果影像尺寸差異很大，你可能需要在 training transforms 中設定適當的 RandCropByPosNegLabeld 尺寸。


In [2]:
%%writefile model.py
import torch
from monai.networks.nets import SwinUNETR
import inspect

def get_swin_unetr_model(img_size=(128, 128, 128), in_channels=1, out_channels=1, feature_size=48):
    """
    建立一個 SwinUNETR 模型。
    Args:
        img_size (tuple): 輸入影像的大小，需與 Dataset 中的 target_shape 一致。
        in_channels (int): 輸入影像的通道數 (例如: T1, T2 雙模態影像就是 2)。
        out_channels (int): 輸出的通道數 (例如: 二元分割就是 1)。
        feature_size (int): Swin Transformer 主幹網路的特徵維度大小。
    Returns:
        torch.nn.Module: SwinUNETR 模型。
    """
    kwargs = dict(
        in_channels=in_channels,
        out_channels=out_channels,
        feature_size=feature_size,
        use_checkpoint=True,  # 使用 Checkpoint 可以節省 GPU 記憶體
    )
    # Some MONAI versions don't support img_size in SwinUNETR __init__.
    if "img_size" in inspect.signature(SwinUNETR).parameters:
        kwargs["img_size"] = img_size

    model = SwinUNETR(**kwargs)
    return model

Writing model.py


In [ ]:
import os
import torch
import numpy as np
from tqdm import tqdm
import glob
from sklearn.model_selection import train_test_split
import sys
import torch.multiprocessing

# MONAI 相關匯入
from monai.data import (
    Dataset,
    decollate_batch,
    MetaTensor
)
from monai.data import DataLoader as MonaiDataLoader
from monai.data.utils import list_data_collate
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
from monai.networks.nets import SwinUNETR
from monai.transforms import (
    AsDiscrete,
    Compose,
    CropForegroundd,
    LoadImaged,
    Orientationd,
    RandFlipd,
    RandCropByPosNegLabeld,
    RandShiftIntensityd,
    ScaleIntensityRangePercentilesd,
    NormalizeIntensityd,
    Spacingd,
    EnsureTyped,
    SpatialPadd,
    RandRotate90d,
    Lambdad,
    DeleteItemsd,
    MapTransform,
    SaveImaged # 匯入 SaveImaged
)
from torch.amp import GradScaler, autocast
from torch.optim.lr_scheduler import ReduceLROnPlateau

# 匯入剛剛寫入的 model.py
from model import get_swin_unetr_model

# ==========================================
#  設定參數區 (取代原本的 argparse)
# ==========================================
class Config:
    # --- 路徑設定 (請修改這裡) ---
    # 假設你的資料在 Drive 的 'MyData/swinMRI/trainData'
    data_dir = '/content/drive/MyDrive/swin_spine/trainData'
    output_dir = '/content/drive/MyDrive/swin_spine/colab_output'
    # 若要接續訓練，填入 .pth 路徑；若要從頭訓練，設為 None 或不存在的路徑
    checkpoint = '/content/drive/MyDrive/swin_spine/best_metric_model0709_resume.pth'

    # --- 模型參數 ---
    img_size = (96, 96, 96)
    out_channels = 21
    feature_size = 48
    use_coord_channels = True  # 是否加入座標通道

    # --- 訓練參數 ---
    learning_rate = 1e-4
    batch_size = 5
    num_samples = 2
    crop_mode = 'label'  # 'label' or 'disc'
    disc_labels = [11, 12, 13, 14, 15]
    pos_ratio = 1
    neg_ratio = 1
    max_train_batches = 0  # 0 表示跑完所有資料
    num_epochs = 100
    val_interval = 5
    num_workers = 0  # Colab 上通常設 2 或 0 比較穩定
    loader = 'torch' # 'torch' or 'thread'

    # --- Class Weights ---
    class_weight_mode = 'disc_boost' # 'none', 'disc_boost', 'invfreq'
    disc_weight = 3.0
    bg_weight = 0.5
    max_class_weight = 20.0

    # --- Learning Rate Scheduler ---
    lr_scheduler = 'plateau'
    lr_factor = 0.5
    lr_patience = 2
    min_lr = 1e-6
    min_delta = 0.001
    early_stop_patience = 0

args = Config()

# ==========================================
#  輔助類別與函數 (保持不變)
# ==========================================

class AddCoordChannelsd(MapTransform):
    def __init__(self, keys, coord_key: str = "coord", allow_missing_keys: bool = False):
        super().__init__(keys, allow_missing_keys)
        self.coord_key = coord_key

    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            if key not in d: continue
            img = d[key]
            if torch.is_tensor(img):
                arr = img
                if arr.ndim == 3: arr = arr.unsqueeze(0)
                _, X, Y, Z = arr.shape
                xs = torch.linspace(-1.0, 1.0, X, dtype=arr.dtype, device=arr.device).view(1, X, 1, 1).expand(1, X, Y, Z)
                ys = torch.linspace(-1.0, 1.0, Y, dtype=arr.dtype, device=arr.device).view(1, 1, Y, 1).expand(1, X, Y, Z)
                zs = torch.linspace(-1.0, 1.0, Z, dtype=arr.dtype, device=arr.device).view(1, 1, 1, Z).expand(1, X, Y, Z)
                coord = torch.cat([xs, ys, zs], dim=0)
            else:
                arr = np.asarray(img)
                if arr.ndim == 3: arr = arr[None, ...]
                _, X, Y, Z = arr.shape
                dtype = arr.dtype if np.issubdtype(arr.dtype, np.floating) else np.float32
                xs = np.linspace(-1.0, 1.0, X, dtype=dtype)[:, None, None]
                ys = np.linspace(-1.0, 1.0, Y, dtype=dtype)[None, :, None]
                zs = np.linspace(-1.0, 1.0, Z, dtype=dtype)[None, None, :]
                xch = np.broadcast_to(xs, (X, Y, Z))
                ych = np.broadcast_to(ys, (X, Y, Z))
                zch = np.broadcast_to(zs, (X, Y, Z))
                coord = np.stack([xch, ych, zch], axis=0)
            d[self.coord_key] = coord
        return d

class ConcatCoordToImaged(MapTransform):
    def __init__(self, image_key: str = "image", coord_key: str = "coord", allow_missing_keys: bool = False):
        super().__init__([image_key, coord_key], allow_missing_keys)
        self.image_key = image_key
        self.coord_key = coord_key

    def __call__(self, data):
        d = dict(data)
        if self.image_key not in d or self.coord_key not in d: return d
        img = d[self.image_key]
        coord = d[self.coord_key]
        if torch.is_tensor(img) and torch.is_tensor(coord):
            d[self.image_key] = torch.cat([img, coord], dim=0)
        else:
            d[self.image_key] = np.concatenate([np.asarray(img), np.asarray(coord)], axis=0)
        return d

class AddDiscMaskd(MapTransform):
    def __init__(self, label_key: str = "label", mask_key: str = "disc_mask", disc_labels=(11, 12, 13, 14, 15), allow_missing_keys: bool = False):
        super().__init__([label_key], allow_missing_keys)
        self.label_key = label_key
        self.mask_key = mask_key
        self.disc_labels = tuple(int(x) for x in disc_labels)

    def __call__(self, data):
        d = dict(data)
        if self.label_key not in d: return d
        lab = d[self.label_key]
        if torch.is_tensor(lab):
            mask = torch.zeros_like(lab, dtype=torch.bool)
            for v in self.disc_labels: mask |= (lab == v)
            d[self.mask_key] = mask.to(lab.dtype)
        else:
            arr = np.asarray(lab)
            mask = np.zeros_like(arr, dtype=bool)
            for v in self.disc_labels: mask |= (arr == v)
            d[self.mask_key] = mask.astype(arr.dtype, copy=False)
        return d

def _load_checkpoint_state(path: str, device: torch.device) -> dict:
    ckpt = torch.load(path, map_location=device)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        ckpt = ckpt["model_state_dict"]
    if not isinstance(ckpt, dict):
        raise ValueError("Checkpoint must be a state_dict dict or contain model_state_dict.")
    ckpt = {k[7:] if k.startswith("module.") else k: v for k, v in ckpt.items()}
    return ckpt

def _inflate_patch_embed_in_channels(state_dict: dict, target_in_channels: int) -> dict:
    if int(target_in_channels) <= 1: return state_dict
    out = dict(state_dict)
    for k, w in state_dict.items():
        if not torch.is_tensor(w) or w.ndim != 5: continue
        if int(w.shape[1]) == int(target_in_channels): continue
        if int(w.shape[1]) == 1:
            new_w = torch.zeros((w.shape[0], int(target_in_channels), *w.shape[2:]), dtype=w.dtype, device=w.device)
            new_w[:, 0, ...] = w[:, 0, ...]
            out[k] = new_w
    return out

# --- 4. Transforms ---
# Move remap_labels_func to global scope
def remap_labels_func(label_tensor):
    # DEVICE needs to be passed or accessed globally if defined before main.
    # For now, let's assume DEVICE is available in the global scope after main is called
    # or we ensure this func is called only after DEVICE is set up.
    # A safer approach might be to pass device as an argument if it's dynamic,
    # but for now, we rely on it being set up.
    device = label_tensor.device
    lookup_table = torch.arange(31, dtype=torch.long, device=device)
    lookup_table[30] = 20
    return lookup_table[label_tensor.long()]

def identity_func(x):
    return x

# ==========================================
#  主執行邏輯 (main)
# ==========================================

def main(args):
    # Set multiprocessing start method for CUDA compatibility
    # This needs to be called once, at the beginning of the program, before any CUDA operations.
    if torch.cuda.is_available():
        torch.multiprocessing.set_start_method('spawn', force=True)

    # --- 1. 設定參數與環境 ---
    DATA_DIR = args.data_dir
    OUTPUT_DIR = args.output_dir
    os.makedirs(OUTPUT_DIR, exist_ok=True) # 確保輸出目錄存在

    if torch.cuda.is_available():
        DEVICE = torch.device("cuda")
        print(f"GPU 設備已找到！將使用: {torch.cuda.get_device_name(0)}")
    else:
        DEVICE = torch.device("cpu")
        print("未找到 GPU 設備，將使用 CPU 進行訓練 (會很慢)。")

    # --- 2. 資料準備 ---
    def create_datalist_from_folders(data_dir: str):
        print(f"從 {data_dir} 掃描 imagesTr/labelsTr...")
        import nibabel as nib

        def stem(path: str) -> str:
            fn = os.path.basename(path)
            if fn.endswith(".nii.gz"): return fn[:-7]
            if fn.endswith(".nii"): return fn[:-4]
            return os.path.splitext(fn)[0]

        def candidates_from_image_stem(s: str):
            yield s
            for suf in ("_t2", "_t1", "_T2", "_T1"):
                if s.endswith(suf): yield s[: -len(suf)]

        image_paths_gz = glob.glob(os.path.join(data_dir, "imagesTr", "*.nii.gz"))
        image_paths_nii = glob.glob(os.path.join(data_dir, "imagesTr", "*.nii"))
        all_image_paths = sorted(image_paths_gz + image_paths_nii)

        label_paths_gz = glob.glob(os.path.join(data_dir, "labelsTr", "*.nii.gz"))
        label_paths_nii = glob.glob(os.path.join(data_dir, "labelsTr", "*.nii"))
        all_label_paths = sorted(label_paths_gz + label_paths_nii)

        label_map = {}
        for p in all_label_paths:
            k = stem(p)
            if k not in label_map or p.endswith(".nii.gz"):
                label_map[k] = p

        datalist = []
        for img_path in all_image_paths:
            img_stem = stem(img_path)
            label_path = None
            for cand in candidates_from_image_stem(img_stem):
                if cand in label_map:
                    label_path = label_map[cand]
                    break
            if not label_path:
                # print(f"警告：找不到圖片 '{img_path}' 的對應標籤，跳過。")
                continue

            # header check (optional, can skip to speed up)
            try:
                # 這裡為了速度可以註解掉，或者保留以確保資料品質
                pass
            except Exception:
                continue

            datalist.append({"image": img_path, "label": label_path})

        return datalist

    all_files = create_datalist_from_folders(DATA_DIR)
    if not all_files:
        print(f"\n錯誤：在 {DATA_DIR} 下沒有成功配對任何圖片和標籤！")
        return

    train_files, val_files = train_test_split(all_files, test_size=0.2, random_state=42)
    print(f"成功配對！訓練集: {len(train_files)} 筆, 驗證集: {len(val_files)} 筆。")

    def compute_class_weights_from_labels(label_paths, out_channels: int) -> np.ndarray:
        import nibabel as nib
        counts = np.zeros((out_channels,), dtype=np.float64)
        print("正在計算 Class Weights (可能需要一點時間)...")
        for p in tqdm(label_paths, desc="Scanning labels"):
            try:
                lab = nib.load(p).get_fdata()
                lab = np.rint(lab).astype(np.int32, copy=False)
                lab[lab == 30] = 20
                lab = np.clip(lab, 0, out_channels - 1)
                bc = np.bincount(lab.reshape(-1), minlength=out_channels).astype(np.float64)
                counts += bc
            except: pass
        counts = np.maximum(counts, 1.0)
        freq = counts / float(counts.sum())
        w = 1.0 / (freq + 1e-12)
        w = w / float(np.mean(w))
        w = np.clip(w, 0.0, float(args.max_class_weight))
        return w.astype(np.float32)

    crop_label_key = "label" if args.crop_mode == "label" else "disc_mask"
    crop_keys = ["image", "label", "coord"] if args.use_coord_channels else ["image", "label"]
    if args.crop_mode == "disc": crop_keys = crop_keys + ["disc_mask"]

    train_transforms = Compose([
        LoadImaged(keys=["image", "label"], reader="NibabelReader", ensure_channel_first=True),
        Lambdad(keys="label", func=remap_labels_func),
        CropForegroundd(keys=["image", "label"], source_key="image"),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(keys=["image", "label"], pixdim=(1.5, 1.5, 1.0), mode=("bilinear", "nearest")),
        ScaleIntensityRangePercentilesd(keys=["image"], lower=1.0, upper=99.0, b_min=0.0, b_max=1.0, clip=True),
        AddCoordChannelsd(keys=["image"], coord_key="coord") if args.use_coord_channels else Lambdad(keys="image", func=identity_func),
        AddDiscMaskd(label_key="label", mask_key="disc_mask", disc_labels=args.disc_labels) if args.crop_mode == "disc" else Lambdad(keys="label", func=identity_func),
        SpatialPadd(keys=crop_keys, spatial_size=args.img_size, method="symmetric"),
        RandCropByPosNegLabeld(keys=crop_keys, label_key=crop_label_key, spatial_size=args.img_size, pos=args.pos_ratio, neg=args.neg_ratio, num_samples=args.num_samples, image_key="image", image_threshold=0),
        RandFlipd(keys=crop_keys, spatial_axis=[0], prob=0.10),
        RandFlipd(keys=crop_keys, spatial_axis=[1], prob=0.10),
        RandFlipd(keys=crop_keys, spatial_axis=[2], prob=0.10),
        RandRotate90d(keys=crop_keys, prob=0.10, max_k=3),
        RandShiftIntensityd(keys=["image"], offsets=0.10, prob=0.50),
        EnsureTyped(keys=crop_keys, device=DEVICE, track_meta=False),
        ConcatCoordToImaged(image_key="image", coord_key="coord") if args.use_coord_channels else Lambdad(keys="image", func=identity_func),
        DeleteItemsd(keys=["coord"]) if args.use_coord_channels else Lambdad(keys="image", func=identity_func),
        DeleteItemsd(keys=["disc_mask"]) if args.crop_mode == "disc" else Lambdad(keys="label", func=identity_func),
    ])

    val_transforms = Compose([
        LoadImaged(keys=["image", "label"], reader="NibabelReader", ensure_channel_first=True),
        Lambdad(keys="label", func=remap_labels_func),
        CropForegroundd(keys=["image", "label"], source_key="image"),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(keys=["image", "label"], pixdim=(1.5, 1.5, 1.0), mode=("bilinear", "nearest")),
        ScaleIntensityRangePercentilesd(keys=["image"], lower=1.0, upper=99.0, b_min=0.0, b_max=1.0, clip=True),
        AddCoordChannelsd(keys=["image"], coord_key="coord") if args.use_coord_channels else Lambdad(keys="image", func=identity_func),
        EnsureTyped(keys=["image", "label", "coord"] if args.use_coord_channels else ["image", "label"], device=DEVICE, track_meta=False),
        ConcatCoordToImaged(image_key="image", coord_key="coord") if args.use_coord_channels else Lambdad(keys="image", func=identity_func),
        DeleteItemsd(keys=["coord"]) if args.use_coord_channels else Lambdad(keys="image", func=identity_func),
    ])

    # --- 5. DataLoader ---
    train_ds = Dataset(data=train_files, transform=train_transforms)
    val_ds = Dataset(data=val_files, transform=val_transforms)

    train_loader = MonaiDataLoader(train_ds, batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers, collate_fn=list_data_collate)
    val_loader = MonaiDataLoader(val_ds, batch_size=1, shuffle=False, num_workers=args.num_workers, collate_fn=list_data_collate)

    # --- 6. Model & Loss ---
    print("正在建立模型...")
    in_ch = 4 if args.use_coord_channels else 1
    model = get_swin_unetr_model(img_size=args.img_size, in_channels=in_ch, out_channels=args.out_channels, feature_size=args.feature_size).to(DEVICE)

    class_weights = None
    if args.class_weight_mode != "none":
        if args.class_weight_mode == "disc_boost":
            w = np.ones((args.out_channels,), dtype=np.float32)
            for idx in (11, 12, 13, 14, 15):
                if idx < args.out_channels: w[idx] = float(args.disc_weight)
            w[0] = float(args.bg_weight)
            class_weights = torch.tensor(w, dtype=torch.float32, device=DEVICE)
            print(f"Class weights (disc_boost): {w}")
        elif args.class_weight_mode == "invfreq":
            label_paths = [d["label"] for d in train_files]
            w = compute_class_weights_from_labels(label_paths, out_channels=args.out_channels)
            for idx in (11, 12, 13, 14, 15):
                if idx < args.out_channels: w[idx] *= float(args.disc_weight)
            w = np.clip(w, 0.0, float(args.max_class_weight))
            class_weights = torch.tensor(w, dtype=torch.float32, device=DEVICE)
            print("Class weights (invfreq) calculated.")

    loss_function = DiceCELoss(to_onehot_y=True, softmax=True, weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, weight_decay=1e-5)

    scheduler = None
    if args.lr_scheduler == "plateau":
        scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=args.lr_factor, patience=args.lr_patience, min_lr=args.min_lr, threshold=args.min_delta, threshold_mode="abs")

    start_epoch = 0
    if args.checkpoint and os.path.exists(args.checkpoint):
        print(f"正在從 {args.checkpoint} 載入權重...")
        try:
            state = _load_checkpoint_state(args.checkpoint, DEVICE)
            if args.use_coord_channels:
                state = _inflate_patch_embed_in_channels(state, target_in_channels=in_ch)
            model.load_state_dict(state)
            print("載入成功！")
        except Exception as e:
            print(f"載入失敗 ({e})，將從頭開始訓練。")

    # --- 7. Training Loop ---
    post_label = AsDiscrete(to_onehot=args.out_channels)
    post_pred = AsDiscrete(argmax=True, to_onehot=args.out_channels)
    dice_metric = DiceMetric(include_background=True, reduction="mean", get_not_nans=False)
    dice_metric_per_class = DiceMetric(include_background=True, reduction="none", get_not_nans=False)

    best_metric = -1
    best_metric_epoch = -1
    no_improve_checks = 0
    scaler = GradScaler(device=DEVICE.type)

    print(f"開始訓練... (Total Epochs: {args.num_epochs})")

    for epoch in range(start_epoch, args.num_epochs):
        model.train()
        epoch_loss = 0
        seen_batches = 0

        # tqdm 在 Colab 有時會太長，我們簡化顯示
        progress_bar = tqdm(train_loader, desc=f"Ep {epoch+1}/{args.num_epochs}", unit="batch", leave=False)

        for batch_data in progress_bar:
            images, labels = batch_data["image"], batch_data["label"]
            optimizer.zero_grad(set_to_none=True)

            with autocast(device_type=DEVICE.type):
                outputs = model(images)
                loss = loss_function(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
            seen_batches += 1
            if args.max_train_batches and seen_batches >= args.max_train_batches: break

        avg_epoch_loss = epoch_loss / (seen_batches if seen_batches else 1)

        # Validation
        if (epoch + 1) % args.val_interval == 0:
            model.eval()
            with torch.no_grad():
                for val_data in val_loader:
                    val_images, val_labels = val_data["image"], val_data["label"]
                    with autocast(device_type=DEVICE.type):
                        val_outputs = sliding_window_inference(val_images, args.img_size, 4, model)

                    val_labels_list = decollate_batch(val_labels)
                    val_labels_convert = [post_label(val_label_tensor) for val_label_tensor in val_labels_list]
                    val_outputs_list = decollate_batch(val_outputs)
                    val_output_convert = [post_pred(val_pred_tensor) for val_pred_tensor in val_outputs_list]

                    dice_metric(y_pred=val_output_convert, y=val_labels_convert)
                    dice_metric_per_class(y_pred=val_output_convert, y=val_labels_convert)

                mean_dice = dice_metric.aggregate().item()
                dice_metric.reset()
                per_class = dice_metric_per_class.aggregate()
                dice_metric_per_class.reset()

                # Calculate Disc Mean (labels 11-15)
                disc_mean = None
                try:
                    pc = per_class.detach().cpu().numpy().astype(np.float32).ravel()
                    disc_idx = [i for i in (11, 12, 13, 14, 15) if i < pc.shape[0]]
                    if disc_idx: disc_mean = float(np.nanmean(pc[disc_idx]))
                except: pass

                # Output Log
                log_str = f"Epoch {epoch+1}: Loss={avg_epoch_loss:.4f}, MeanDice={mean_dice:.4f}"
                if disc_mean is not None: log_str += f", DiscDice={disc_mean:.4f}"
                print(log_str)

                # Save Best Model
                metric_for_best = disc_mean if (disc_mean is not None) else mean_dice
                if metric_for_best > (best_metric + args.min_delta):
                    best_metric = metric_for_best
                    best_metric_epoch = epoch + 1
                    no_improve_checks = 0
                    save_path = os.path.join(OUTPUT_DIR, "best_metric_model.pth")
                    torch.save(model.state_dict(), save_path)
                    print(f"  >>> New Best! Saved to {save_path}")
                else:
                    no_improve_checks += 1

                # Scheduler Step
                if scheduler is not None:
                    scheduler.step(metric_for_best)
                    if hasattr(optimizer, 'param_groups'):
                        print(f"  LR: {optimizer.param_groups[0]['lr']:.2e}")

                # Early Stop
                if args.early_stop_patience > 0 and no_improve_checks >= args.early_stop_patience:
                    print(f"Early stopping triggered.")
                    break
        else:
            print(f"Epoch {epoch+1}: Loss={avg_epoch_loss:.4f}")

    print(f"\n訓練結束。最佳 Epoch: {best_metric_epoch}, 最佳 Metric: {best_metric:.4f}")
# if __name__ == "__main__":
#     main(args)


IndentationError: unexpected indent (ipython-input-54216237.py, line 351)

In [ ]:
import nibabel as nib
import numpy as np
import os
import matplotlib.pyplot as plt # 導入 matplotlib 用於儲存 PNG

if len(predictions_list) == 0:
    print("沒有找到任何預測結果檔案，無法提取中間切片。請先執行推理。")
else:
    # 選擇第一個預測結果檔案
    predicted_nii_path = predictions_list[0]
    print(f"\n正在從預測結果檔案中提取中間切片: {predicted_nii_path}")

    # 載入 NIfTI 影像
    predicted_img = nib.load(predicted_nii_path)
    predicted_data = predicted_img.get_fdata()
    original_affine = predicted_img.affine

    # 獲取影像的空間形狀
    spatial_shape = predicted_data.shape

    if len(spatial_shape) < 2:
        print(f"影像形狀 {spatial_shape} 太小，無法提取切片。")
    else:
        # 找到張數最少的維度及其索引
        min_dim_size = np.inf
        min_dim_idx = -1
        for i, size in enumerate(spatial_shape):
            if size < min_dim_size:
                min_dim_size = size
                min_dim_idx = i

        if min_dim_idx == -1:
            print("無法確定最小維度。")
        else:
            # 計算該維度的中間切片索引
            middle_slice_idx = spatial_dim = spatial_shape[min_dim_idx] // 2
            print(f"影像形狀: {spatial_shape}, 最小維度是 {min_dim_idx} (大小: {min_dim_size})。將取出第 {middle_slice_idx} 張切片。電視： {middle_slice_data.shape}")

            # 提取中間切片
            slice_selection = [slice(None)] * len(spatial_shape)
            slice_selection[min_dim_idx] = middle_slice_idx
            middle_slice_data = predicted_data[tuple(slice_selection)]

            # 儲存中間切片為 PNG 檔案
            output_slice_filename = os.path.basename(predicted_nii_path).replace('.nii.gz', f'_slice_dim{min_dim_idx}_idx{middle_slice_idx}.png').replace('.nii', f'_slice_dim{min_dim_idx}_idx{middle_slice_idx}.png')
            output_slice_path = os.path.join(inference_args.prediction_output_dir, output_slice_filename)

            # 對影像數據進行歸一化到 0-1 區間，以便 plt.imsave 正確顯示
            # 假設分割結果是整數標籤，我們可以將其直接映射到視覺可區分的灰度值或彩色
            # 這裡簡單歸一化到 0-1
            if middle_slice_data.max() > 0:
                normalized_slice = middle_slice_data / middle_slice_data.max()
            else:
                normalized_slice = middle_slice_data # 如果全為0，則保持不變

            plt.imsave(output_slice_path, normalized_slice, cmap='gray') # 使用灰度色彩映射儲存
            print(f"中間切片已儲存到: {output_slice_path}")


沒有找到任何預測結果檔案，無法提取中間切片。請先執行推理。


### 8. 推理與預測

此部分將使用訓練好的最佳模型（`best_metric_model.pth`）對新的影像資料進行推理，並生成分割結果。請確保您的新影像資料位於 `args.inference_data_dir` 指定的路徑下。

In [ ]:
# ==========================================
#  推理設定區 (請修改這裡)
# ==========================================
class InferenceConfig(Config):
    inference_data_dir = '/content/drive/MyDrive/swin_spine/trainData'
    prediction_output_dir = '/content/drive/MyDrive/swin_spine/colab_output/inference_output'
    best_model_path = '/content/drive/MyDrive/swin_spine/colab_output/best_metric_model.pth'

    # 您也可以在這裡覆寫其他訓練參數，例如 img_size, out_channels 等，
    # 但通常建議保持與訓練時一致。

inference_args = InferenceConfig()

# 確保輸出目錄存在
os.makedirs(inference_args.prediction_output_dir, exist_ok=True)

# 設定設備
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"GPU 設備已找到！將使用: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("未找到 GPU 設備，將使用 CPU 進行推理 (會很慢)。")


GPU 設備已找到！將使用: NVIDIA A100-SXM4-40GB


In [ ]:
# ==========================================
#  資料準備 (推理)
# ==========================================
import nibabel as nib

def create_inference_datalist(inference_data_dir: str):
    print(f"從 {inference_data_dir} 掃描影像檔...")
    image_paths_gz = glob.glob(os.path.join(inference_data_dir, "*.nii.gz"))
    image_paths_nii = glob.glob(os.path.join(inference_data_dir, "*.nii"))
    all_image_paths = sorted(image_paths_gz + image_paths_nii)

    datalist = []
    for img_path in all_image_paths:
        datalist.append({"image": img_path})
    return datalist

inference_files = create_inference_datalist(inference_args.inference_data_dir)

if not inference_files:
    print(f"錯誤：在 {inference_args.inference_data_dir} 下沒有找到任何影像檔！請檢查路徑。")
else:
    print(f"找到 {len(inference_files)} 筆影像資料用於推理。")

# ==========================================
#  Transforms (推理)
# ==========================================
inference_transforms = Compose([
    LoadImaged(keys=["image"], reader="NibabelReader", ensure_channel_first=True),
    CropForegroundd(keys=["image"], source_key="image"),
    Orientationd(keys=["image"], axcodes="RAS"),
    Spacingd(keys=["image"], pixdim=(1.5, 1.5, 1.0), mode="bilinear"),
    ScaleIntensityRangePercentilesd(keys=["image"], lower=1.0, upper=99.0, b_min=0.0, b_max=1.0, clip=True),
    AddCoordChannelsd(keys=["image"], coord_key="coord") if inference_args.use_coord_channels else Lambdad(keys="image", func=identity_func),
    EnsureTyped(keys=["image", "coord"] if inference_args.use_coord_channels else ["image"], device=DEVICE, track_meta=True), # 修改這裡：track_meta=True
    ConcatCoordToImaged(image_key="image", coord_key="coord") if inference_args.use_coord_channels else Lambdad(keys="image", func=identity_func),
    DeleteItemsd(keys=["coord"]) if inference_args.use_coord_channels else Lambdad(keys="image", func=identity_func),
])

inference_ds = Dataset(data=inference_files, transform=inference_transforms)
inference_loader = MonaiDataLoader(inference_ds, batch_size=1, shuffle=False, num_workers=inference_args.num_workers, collate_fn=list_data_collate)


從 /content/drive/MyDrive/swin_spine/trainData 掃描影像檔...
找到 1 筆影像資料用於推理。


/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [ ]:
# ==========================================
#  模型載入與推理
# ==========================================
print("正在載入最佳模型...")
in_ch = 4 if inference_args.use_coord_channels else 1
model = get_swin_unetr_model(img_size=inference_args.img_size, in_channels=in_ch, out_channels=inference_args.out_channels, feature_size=inference_args.feature_size).to(DEVICE)

if os.path.exists(inference_args.best_model_path):
    try:
        state = _load_checkpoint_state(inference_args.best_model_path, DEVICE)
        if inference_args.use_coord_channels:
            state = _inflate_patch_embed_in_channels(state, target_in_channels=in_ch)
        model.load_state_dict(state)
        print(f"成功從 {inference_args.best_model_path} 載入模型權重！")
    except Exception as e:
        print(f"錯誤：無法載入模型權重 ({e})。請檢查路徑和檔案是否正確，或模型架構是否匹配。")
        # 如果載入失敗，可以選擇退出或繼續但結果可能不佳
        exit()
else:
    print(f"錯誤：找不到最佳模型權重檔於 {inference_args.best_model_path}。請確認路徑或先進行訓練。")
    exit()

model.eval()
predictions_list = []

# 初始化 SaveImaged 轉換，用於將預測結果重新採樣回原始影像空間並儲存
# MONAI 的 SaveImaged 在 resample=True 時會使用原始影像的 metadata 進行 resampling
saver = SaveImaged(
    output_dir=inference_args.prediction_output_dir,
    output_postfix="pred",
    resample=True,  # 關鍵：自動重新採樣到原始影像的幾何形狀
    separate_input_and_output=False, # 確保使用原始影像的 metadata
    output_ext=".nii.gz",
    data_root_dir=inference_args.inference_data_dir, # 用於解析原始檔案路徑
    mode="nearest", # 對於離散標籤，使用最近鄰插值
)

with torch.no_grad():
    for idx, inference_data in enumerate(tqdm(inference_loader, desc="進行推理")):
        images = inference_data["image"]
        # 原始影像的 filename_or_path 仍然需要從 inference_files 列表中獲取
        filename_or_path = inference_files[idx]["image"]

        with autocast(device_type=DEVICE.type):
            # 使用 sliding_window_inference 進行預測，以處理大影像
            pred_logits = sliding_window_inference(images, inference_args.img_size, 4, model, overlap=0.5)

        # 將預測結果從 logits 轉換為機率，並取 argmax 得到類別
        pred_softmax = torch.softmax(pred_logits, dim=1)
        pred_label = torch.argmax(pred_softmax, dim=1, keepdim=True)

        # 將 pred_label 轉換為 MetaTensor，並傳遞原始影像的 metadata
        # 這是 SaveImaged(resample=True) 能夠正常工作的前提
        # images.meta 中包含了 LoadImaged 載入時的 original_affine, original_spatial_shape 等資訊
        pred_label_metatensor = MetaTensor(pred_label.cpu(), meta=images.meta.copy())

        # 儲存預測結果
        # saver 會根據 pred_label_metatensor 中的 metadata 和 `resample=True` 進行重新採樣和儲存
        saver({"image": pred_label_metatensor})

        # 構建預期儲存路徑並加入列表
        output_filename = os.path.basename(filename_or_path).replace('.nii.gz', '_pred.nii.gz').replace('.nii', '_pred.nii')
        output_path = os.path.join(inference_args.prediction_output_dir, output_filename)
        predictions_list.append(output_path)

print("推理完成！所有預測結果已儲存到：")
for p in predictions_list:
    print(f"- {p}")


正在載入最佳模型...
成功從 /content/drive/MyDrive/swin_spine/colab_output/best_metric_model.pth 載入模型權重！


NameError: name 'SaveImaged' is not defined